In [474]:
import numpy as np
import pandas as pd
import torch

Read CSV file

In [493]:
df = pd.read_csv('train.csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


Preprocess data

In [494]:
modes = df.mode().iloc[0]
df.fillna(modes, inplace = True)

df['Fare'] = np.log(df['Fare'] + 1)
df = pd.get_dummies(df, columns = ['Sex', 'Pclass', 'Embarked'], dtype = int)

In [498]:
dummy_cols = ['Sex_male', 'Sex_female', 'Pclass_1', 'Pclass_2', 'Pclass_3', 'Embarked_C', 'Embarked_Q', 'Embarked_S']
ind_cols = ['Age', 'SibSp', 'Parch', 'Fare'] + dummy_cols
dep_cols = ['Survived']

Tensor conversion

In [496]:
ind_tensor = torch.tensor(df[ind_cols].values, dtype = torch.float)
dep_tensor = torch.tensor(df[dep_cols].values, dtype = torch.float)

In [497]:
dep_tensor.shape

torch.Size([891, 1])

In [500]:
dep_tensor[:, None].shape

torch.Size([891, 1, 1])

Normalize Tensors

In [479]:
ind_tensor = ind_tensor / ind_tensor.max(dim = 0).values

Parameters

In [480]:
params = torch.rand(ind_tensor.shape[1]) - 0.5
params.requires_grad_(True)

tensor([ 0.0198, -0.0783, -0.4223,  0.1333,  0.4300,  0.3614, -0.1893,  0.4888,
         0.3085, -0.3281, -0.4855,  0.2612], requires_grad=True)

Loss (MAE - Mean Absolute Error)

In [481]:
def cal_preds(params, ind_tensor):
    return (torch.sigmoid((ind_tensor @ params)))

def cal_loss(params, ind_tensor):
    preds = cal_preds(params, ind_tensor)
    loss = torch.abs(preds - dep_tensor).mean()
    return loss

In [482]:
def optimize_params(params, lr = 0.1):
    with torch.no_grad():
        params.sub_(params.grad * 0.1)
        params.grad.zero_()

Hyperparameter

In [483]:
lr = 0.5
epochs = 100

Train model

In [ ]:
def train_model(params, ind_tensor, epochs, lr):
    for epoch in range(epochs):
        loss = cal_loss(params, ind_tensor)
        loss.backward()
        optimize_params(params, lr)
        print(f'{epoch + 1}. {loss.item():.4f}')
    
    return params

1. 0.5406
2. 0.5401
3. 0.5397
4. 0.5392
5. 0.5388
6. 0.5383
7. 0.5378
8. 0.5374
9. 0.5369
10. 0.5365
11. 0.5360
12. 0.5355
13. 0.5350
14. 0.5346
15. 0.5341
16. 0.5336
17. 0.5331
18. 0.5326
19. 0.5321
20. 0.5316
21. 0.5311
22. 0.5306
23. 0.5301
24. 0.5296
25. 0.5291
26. 0.5286
27. 0.5281
28. 0.5276
29. 0.5270
30. 0.5265
31. 0.5260
32. 0.5255
33. 0.5249
34. 0.5244
35. 0.5239
36. 0.5233
37. 0.5228
38. 0.5222
39. 0.5217
40. 0.5211
41. 0.5206
42. 0.5200
43. 0.5195
44. 0.5189
45. 0.5183
46. 0.5178
47. 0.5172
48. 0.5167
49. 0.5161
50. 0.5155
51. 0.5149
52. 0.5144
53. 0.5138
54. 0.5132
55. 0.5126
56. 0.5120
57. 0.5115
58. 0.5109
59. 0.5103
60. 0.5097
61. 0.5091
62. 0.5085
63. 0.5079
64. 0.5073
65. 0.5067
66. 0.5062
67. 0.5056
68. 0.5050
69. 0.5044
70. 0.5038
71. 0.5032
72. 0.5026
73. 0.5020
74. 0.5014
75. 0.5008
76. 0.5002
77. 0.4996
78. 0.4989
79. 0.4983
80. 0.4977
81. 0.4971
82. 0.4965
83. 0.4959
84. 0.4953
85. 0.4947
86. 0.4941
87. 0.4935
88. 0.4929
89. 0.4923
90. 0.4917
91. 0.4911
92. 0.49

Accuracy

In [ ]:
params = train_model(params, ind_tensor, epochs, lr)
preds = (cal_preds(params, ind_tensor) > 0.5).int()
preds

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0,
        1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1,
        0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0,
        0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1,
        0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [486]:
accuracy = (dep_tensor== preds).float().mean().item()
print(f'Accuracy: {accuracy:.4f}')

Accuracy: 0.5726


Evaluation

In [487]:
test_df = pd.read_csv('test.csv')

test_df.fillna(modes, inplace = True)

test_df = pd.get_dummies(test_df, columns = ['Sex', 'Pclass', 'Embarked'], dtype = int)
test_df['Fare'] = np.log(test_df['Fare'] + 1)

test_ind_tensor = torch.tensor(test_df[ind_cols].values, dtype = torch.float)
test_ind_tensor = test_ind_tensor / ind_tensor.max(dim = 0).values

preds = cal_preds(params, test_ind_tensor)

test_df['Survived'] = (preds > 0.5).int()

test_df.loc[test_df['Survived'] == 1, ['PassengerId', 'Survived']]

,PassengerId,Survived


In [492]:
dep_tensor = dep_tensor[:, None]
dep_tensor.shape

torch.Size([891, 1, 1, 1, 1])

In [491]:
ind_tensor = ind_tensor[:, None]
ind_tensor.shape

torch.Size([891, 1, 1, 12])